In [1]:
import subprocess
from pathlib import Path
import re

repo_path = Path('/workspaces/-294612-git-.github.io')

def check_html_syntax(file_path):
    """Basic HTML syntax check"""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()

        # Check for basic HTML structure
        if not content.strip().startswith('<!DOCTYPE html>'):
            return f"Missing DOCTYPE declaration"

        if '<html' not in content or '</html>' not in content:
            return f"Missing html tags"

        if '<head>' not in content or '</head>' not in content:
            return f"Missing head tags"

        if '<body>' not in content or '</body>' not in content:
            return f"Missing body tags"

        # Check for unclosed tags (basic check)
        open_tags = []
        for match in re.finditer(r'<(/?)(\w+)[^>]*>', content):
            is_closing = match.group(1) == '/'
            tag = match.group(2).lower()

            if tag in ['br', 'img', 'input', 'meta', 'link', 'hr']:  # self-closing tags
                continue

            if not is_closing:
                open_tags.append(tag)
            elif open_tags and open_tags[-1] == tag:
                open_tags.pop()
            else:
                return f"Unmatched closing tag: </{tag}>"

        if open_tags:
            return f"Unclosed tags: {open_tags}"

        return "OK"

    except Exception as e:
        return f"Error reading file: {e}"

def validate_files():
    """Validate all HTML files in the repository"""
    html_files = []
    for ext in ['*.html', '**/*.html']:
        html_files.extend(repo_path.glob(ext))

    print(f"Found {len(html_files)} HTML files to validate")

    errors = []
    for file_path in sorted(html_files):
        rel_path = file_path.relative_to(repo_path)
        result = check_html_syntax(file_path)
        if result != "OK":
            errors.append(f"{rel_path}: {result}")
            print(f"❌ {rel_path}: {result}")
        else:
            print(f"✅ {rel_path}: OK")

    return errors

# Check git status
print("=== GIT STATUS CHECK ===")
result = subprocess.run(['git', 'status', '--porcelain'], cwd=repo_path, capture_output=True, text=True)
status_lines = [line for line in result.stdout.strip().split('\n') if line.strip()]

if status_lines:
    print("Files with changes:")
    for line in status_lines:
        status = line[:2]
        filename = line[2:].strip()
        print(f"  {status} {filename}")
else:
    print("No changes detected")

print("\n=== HTML VALIDATION ===")
errors = validate_files()

print(f"\n=== SUMMARY ===")
if errors:
    print(f"❌ Found {len(errors)} HTML validation errors")
    for error in errors[:5]:  # Show first 5 errors
        print(f"  {error}")
    if len(errors) > 5:
        print(f"  ... and {len(errors) - 5} more")
else:
    print("✅ All HTML files passed basic validation")

if status_lines:
    print(f"\n📝 {len(status_lines)} files ready to commit")
else:
    print("\n✨ No files to commit")

=== GIT STATUS CHECK ===
Files with changes:
  A  "grade 6/english/grade6-english-topics.html"
  A  "grade 6/english/grade6-grammar-usage.html"
  A  "grade 6/english/grade6-reading-analysis.html"
  A  "grade 6/english/grade6-vocabulary.html"
  A  "grade 6/history/grade6-ancient-civilizations.html"
  A  "grade 6/history/grade6-civics-economics.html"
  A  "grade 6/history/grade6-geography-culture.html"
  A  "grade 6/history/grade6-history-topics.html"
  A  "grade 6/math/grade6-equations-inequalities.html"
  A  "grade 6/math/grade6-geometry-statistics.html"
  A  "grade 6/math/grade6-math-topics.html"
  A  "grade 6/math/grade6-ratios-expressions.html"
  A  "grade 6/science/grade6-cells-life.html"
  A  "grade 6/science/grade6-chemistry-physics.html"
  A  "grade 6/science/grade6-earth-atmosphere.html"
  A  "grade 6/science/grade6-science-topics.html"
  R  grade2/science/grade2-plants.html.html -> grade2/grade2-animals.html
  A  grade2/grade2-economics.html
  A  grade2/grade2-english-topics.h